In [2]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
import pickle
import time

options = Options()
options.add_argument("--start-maximized")
options.add_argument("window-size=1920x1080")
options.add_argument("lang=ko_KR")

service = Service(ChromeDriverManager().install())

driver = webdriver.Chrome(service=service, options=options)

driver.get("https://www.instagram.com/accounts/login/")
print("수동으로 인증 시작!")
time.sleep(30)

with open("instagram_cookies.pkl", "wb") as f :
    pickle.dump(driver.get_cookies(), f)
print("쿠키 저장 완료")
driver.quit()


# cookie, wb 형태로 저장


수동으로 인증 시작!
쿠키 저장 완료


In [4]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
import pickle
import time

options = Options()
options.add_argument("--start-maximized")
options.add_argument("window-size=1920x1080")
options.add_argument("lang=ko_KR")

service = Service(ChromeDriverManager().install())

driver = webdriver.Chrome(service=service, options=options)

driver.get("https://www.instagram.com/")
with open("instagram_cookies.pkl", "rb") as f :
    cookies = pickle.load(f)
for cookie in cookies :
    driver.add_cookie(cookie)

driver.refresh()
time.sleep(5)

try :
    not_now = driver.find_element(By.XPATH, "//button[text()='나중에 하기']")
    not_now.click()
    time.sleep(2)
except :
    pass

keyword = "도시락"
driver.get(f"https://www.instagram.com/explore/tags/{keyword}")

post_links = set()
for _ in range(3) :
    links = driver.find_elements(By.TAG_NAME, "a")
    for link in links :
        href = link.get_attribute("href")
        if href and "/p/" in href :
            post_links.add(href)
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(5)

results = []
for url in list(post_links)[:10] :
    driver.get(url)
    time.sleep(3)
    caption = ""

    try :
        img_tag = driver.find_element(By.XPATH, "//img[@alt]")
        caption = img_tag.get_attribute("alt").strip()
        results.append({"url" : url, "caption" : caption})
    except Exception as e :
        print(f"캡션 추출을 실패 : {url} | {e}")
        continue

driver.quit()

for r in results :
    print(r["url"])
    print(r["caption"])
    print("="*60)
    

캡션 추출을 실패 : https://www.instagram.com/p/C4_raWfSDCq/ | Message: no such element: Unable to locate element: {"method":"xpath","selector":"//img[@alt]"}
  (Session info: chrome=141.0.7390.55); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#nosuchelementexception
Stacktrace:
	GetHandleVerifier [0x0x106fe43+66515]
	GetHandleVerifier [0x0x106fe84+66580]
	(No symbol) [0x0xe5dc48]
	(No symbol) [0x0xea8704]
	(No symbol) [0x0xea8aab]
	(No symbol) [0x0xeef482]
	(No symbol) [0x0xecb214]
	(No symbol) [0x0xeecba7]
	(No symbol) [0x0xecafc6]
	(No symbol) [0x0xe9c2ca]
	(No symbol) [0x0xe9d154]
	GetHandleVerifier [0x0x12c7353+2521315]
	GetHandleVerifier [0x0x12c22d3+2500707]
	GetHandleVerifier [0x0x1097c94+229924]
	GetHandleVerifier [0x0x10881f8+165768]
	GetHandleVerifier [0x0x108ecad+193085]
	GetHandleVerifier [0x0x1078158+100072]
	GetHandleVerifier [0x0x10782f0+100480]
	GetHandleVerifier [0x0x10625aa+11066]
	BaseThreadInitThunk [

InvalidSessionIdException: Message: invalid session id; For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#invalidsessionidexception
Stacktrace:
	GetHandleVerifier [0x0x106fe43+66515]
	GetHandleVerifier [0x0x106fe84+66580]
	(No symbol) [0x0xe5da6b]
	(No symbol) [0x0xe9b5ab]
	(No symbol) [0x0xecb086]
	(No symbol) [0x0xec667c]
	(No symbol) [0x0xec5bf3]
	(No symbol) [0x0xe2e72d]
	(No symbol) [0x0xe2ecae]
	(No symbol) [0x0xe2f14d]
	GetHandleVerifier [0x0x12c7353+2521315]
	GetHandleVerifier [0x0x12c22d3+2500707]
	GetHandleVerifier [0x0x1097c94+229924]
	GetHandleVerifier [0x0x10881f8+165768]
	GetHandleVerifier [0x0x108ecad+193085]
	(No symbol) [0x0xe2e3e3]
	(No symbol) [0x0xe2db60]
	GetHandleVerifier [0x0x140c20f+3852191]
	BaseThreadInitThunk [0x0x76d35d49+25]
	RtlInitializeExceptionChain [0x0x77add6db+107]
	RtlGetAppContainerNamedObjectPath [0x0x77add661+561]


In [5]:
# -*- coding: utf-8 -*-
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.common.keys import Keys
from selenium.common.exceptions import TimeoutException, WebDriverException
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

import pickle, time, random, re, csv
from urllib.parse import quote

# ==================== 설정 ====================
KEYWORD          = "도시락"
TARGET_CAPTIONS  = 2             # 데모값(실전은 520 권장)
MAX_SCROLLS      = 2             # 데모값(실전은 3000 권장)
VISIT_WAIT       = (1.8, 3.4)
SCROLL_SHORT     = (0.15, 0.35)
SCROLL_PAUSE     = (0.9, 1.8)
CSV_PATH         = "instagram_captions.csv"
INSTA_COOKIE_PKL = "instagram_cookies.pkl"
# =================================================

def make_driver():
    options = Options()
    options.add_argument("--start-maximized")
    options.add_argument("window-size=1920x1080")
    options.add_argument("lang=ko_KR")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--disable-infobars")
    service = Service(ChromeDriverManager().install())
    drv = webdriver.Chrome(service=service, options=options)
    drv.set_page_load_timeout(60)
    return drv

driver = make_driver()
wait = WebDriverWait(driver, 12)
actions = ActionChains(driver)

# ---------------- 로그인/쿠키 ----------------
driver.get("https://www.instagram.com/")
time.sleep(2)
try:
    with open(INSTA_COOKIE_PKL, "rb") as f:
        cookies = pickle.load(f)
    for c in cookies:
        if "expiry" in c and isinstance(c["expiry"], float):
            c["expiry"] = int(c["expiry"])
        try:
            driver.add_cookie(c)
        except Exception:
            pass
    driver.refresh()
    time.sleep(3)
except FileNotFoundError:
    print("[경고] instagram_cookies.pkl이 없습니다. 로그인 상태가 아닐 수 있습니다.")

# 팝업 무시(있으면)
for txt in ["필수 쿠키만 허용", "쿠키 허용", "Only allow essential cookies", "Allow all cookies",
            "나중에 하기", "Not now", "지금 아니오"]:
    try:
        btn = driver.find_element(By.XPATH, f"//button[normalize-space()='{txt}']")
        driver.execute_script("arguments[0].click();", btn)
        time.sleep(0.3)
    except:
        pass

# ---------------- 태그 페이지 진입 ----------------
def goto_hashtag(tag):
    enc = quote(tag)
    url1 = f"https://www.instagram.com/explore/tags/{enc}/"
    driver.get(url1)
    try:
        wait.until(EC.presence_of_element_located((By.TAG_NAME, "a")))
        return
    except TimeoutException:
        pass
    url2 = f"https://www.instagram.com/explore/tags/{tag}"
    driver.get(url2)
    wait.until(EC.presence_of_element_located((By.TAG_NAME, "a")))
goto_hashtag(KEYWORD)

# ---------------- 자연스러운 스크롤 ----------------
def human_scroll_cycle():
    actions.send_keys(Keys.END).perform()
    time.sleep(random.uniform(*SCROLL_PAUSE))
    for _ in range(random.randint(2, 4)):
        dy = random.randint(400, 1100)
        driver.execute_script("window.scrollBy(0, arguments[0]);", dy)
        time.sleep(random.uniform(*SCROLL_SHORT))
        if random.random() < 0.22:
            driver.execute_script("window.scrollBy(0, arguments[0]);", -random.randint(120, 260))
            time.sleep(random.uniform(0.1, 0.25))

# ---------------- 링크 수집 ----------------
post_links, seen = [], set()
prev_cnt, stale_rounds, scrolls = 0, 0, 0

while len(post_links) < int(TARGET_CAPTIONS * 1.2) and scrolls < MAX_SCROLLS and stale_rounds < 8:
    anchors = driver.find_elements(By.CSS_SELECTOR, "a[href*='/p/'], a[href*='/reel/']")
    for a in anchors:
        href = a.get_attribute("href")
        if href and ("/p/" in href or "/reel/" in href) and href not in seen:
            seen.add(href)
            post_links.append(href)

    human_scroll_cycle()
    scrolls += 1

    if len(post_links) == prev_cnt:
        stale_rounds += 1
    else:
        stale_rounds = 0
        prev_cnt = len(post_links)

    if scrolls % 5 == 0:
        print(f"[스크롤 {scrolls}] 링크 {len(post_links)}개 수집")
print(f"[링크 수집 완료] 총 {len(post_links)}개")

# ---------------- 공통: 프레임/문서 어디서든 캡션/태그 JS 추출 ----------------
JS_EXTRACT = r"""
function extractFrom(doc) {
  function txt(el){ return el ? (el.textContent || '').trim() : ''; }
  // article 위치(모달 포함)
  var art = doc.querySelector('div[role="dialog"] article') || doc.querySelector('article');
  if(!art) return {caption:'', hashtags:[]};

  // 더보기 클릭(있으면)
  var more = Array.from(art.querySelectorAll('span,button')).find(e=>{
    let t=(e.textContent||'').trim();
    return t==='더 보기'||t==='더보기'||t==='More'||t==='more'||t==='… 더보기'||t==='… more';
  });
  if(more){ more.click(); }

  // 캡션 블록: 첫 li
  var li1 = art.querySelector('ul li:first-child');
  var capNode = (li1 && (li1.querySelector('h1[dir="auto"]') || li1.querySelector('span[dir="auto"]'))) 
                || art.querySelector('h1[dir="auto"]');
  var caption = txt(capNode);
  if(!caption && li1) caption = txt(li1);

  // 해시태그: 같은 블록 안 a[href^="/explore/tags/"]
  var seen = new Set(); var tags = [];
  if(li1){
    li1.querySelectorAll('a[href^="/explore/tags/"]').forEach(a=>{
      var raw = (a.textContent||'').trim().replace(/^#\s*/,'')
                   .replace(/[\u200b-\u200f\u202a-\u202e\u2060\s]+/g,'');
      if(raw && !seen.has(raw)){ seen.add(raw); tags.push(raw); }
    });
  }
  // 보조: 캡션 문자열에서 추출
  if(!tags.length && caption){
    (caption.match(/#([0-9A-Za-z_가-힣_]+)/g)||[]).forEach(s=>{
      var t = s.slice(1);
      if(t && !seen.has(t)){ seen.add(t); tags.push(t); }
    });
  }
  return {caption: caption, hashtags: tags};
}
// 기본 문서에서 시도
var data = extractFrom(document);
return data;
"""

def extract_from_any_frame():
    """기본 문서 → 모든 iframe 순회하며 JS로 직접 추출"""
    # 1) 기본 문서
    try:
        data = driver.execute_script(JS_EXTRACT)
        if (data and (data.get("caption") or data.get("hashtags"))):
            return data.get("caption", ""), data.get("hashtags", [])
    except WebDriverException:
        pass

    # 2) 모든 iframe 순회
    frames = driver.find_elements(By.TAG_NAME, "iframe")
    for fr in frames:
        try:
            driver.switch_to.frame(fr)
            data = driver.execute_script(JS_EXTRACT)
            driver.switch_to.default_content()
            if (data and (data.get("caption") or data.get("hashtags"))):
                return data.get("caption", ""), data.get("hashtags", [])
        except WebDriverException:
            try:
                driver.switch_to.default_content()
            except:
                pass
            continue

    # 3) 실패 시 빈값
    try:
        driver.switch_to.default_content()
    except:
        pass
    return "", []

def wait_article_any_frame(timeout=12):
    """어느 프레임이든 article이 등장할 때까지 폴링"""
    end = time.time() + timeout
    while time.time() < end:
        try:
            # 기본 문서
            found = driver.execute_script(
                "return !!(document.querySelector('div[role=\"dialog\"] article') || document.querySelector('article'));"
            )
            if found: return True
        except WebDriverException:
            pass
        # 프레임 폴링
        frames = driver.find_elements(By.TAG_NAME, "iframe")
        for fr in frames:
            try:
                driver.switch_to.frame(fr)
                found = driver.execute_script("return !!document.querySelector('article');")
                driver.switch_to.default_content()
                if found: return True
            except WebDriverException:
                try:
                    driver.switch_to.default_content()
                except:
                    pass
        time.sleep(0.35)
    return False

# ---------------- 캡션/해시태그 추출 루프 ----------------
results = []
for i, url in enumerate(post_links):
    if len(results) >= TARGET_CAPTIONS:
        break
    try:
        driver.get(url)
        time.sleep(random.uniform(*VISIT_WAIT))
        # 프레임 어디든 article 대기
        ok = wait_article_any_frame(timeout=12)
        if not ok:
            # 한 번 더 새로고침 후 재시도
            driver.refresh()
            time.sleep(random.uniform(*VISIT_WAIT))
            wait_article_any_frame(timeout=12)

        cap, tags = extract_from_any_frame()
        # 마지막 백업: meta og:description
        if not cap:
            try:
                meta = driver.find_element(By.CSS_SELECTOR, 'meta[property="og:description"]')
                cap = (meta.get_attribute("content") or "").strip()
            except:
                pass
        results.append({"url": url, "caption": cap, "hashtags": tags})
        if (i + 1) % 20 == 0:
            print(f"[진행] {i+1} 방문 / {len(results)} 수집")
    except Exception as e:
        print(f"[오류] {url} 처리 실패: {e}")
        # 페이지 충돌/크래시 방지: 컨텐츠 초기화 후 다음으로
        try:
            driver.switch_to.default_content()
        except:
            pass
        continue

# ---------------- CSV 저장 ----------------
with open(CSV_PATH, "w", newline="", encoding="utf-8-sig") as f:
    w = csv.writer(f)
    w.writerow(["url", "caption", "hashtags"])
    for r in results:
        w.writerow([r["url"], r["caption"], ";".join(r["hashtags"])])

print(f"[CSV 저장 완료] {CSV_PATH} (총 {len(results)}건)")
driver.quit()


[링크 수집 완료] 총 0개
[CSV 저장 완료] instagram_captions.csv (총 0건)
